# 08 · Hybrid Retrieval：向量检索 + BM25

向量检索擅长找“意思相近”的内容，BM25 擅长命中 SKU、货号和专有词。混合检索把两路结果都保留下来，再用 RRF（倒数排名融合）合并排名。

In [ ]:
import math
import os
import re
from collections import Counter, defaultdict

import numpy as np
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

import os
import re
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-zh-v1.5")
model = SentenceTransformer(EMBEDDING_MODEL)

def split_markdown(text, max_chars=800):
    sections = re.split(r"\n(?=#{1,3}\s)", text)
    chunks, current = [], ""
    for section in sections:
        if current and len(current) + len(section) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += section + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for text in split_markdown(path.read_text(encoding="utf-8")):
            items.append({"source": str(path.relative_to(data_dir)), "text": text})
    return items

def build_index(chunks):
    vectors = model.encode([item["text"] for item in chunks], normalize_embeddings=True)
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="fashion_knowledge",
        vectors_config=models.VectorParams(size=vectors.shape[1], distance=models.Distance.COSINE),
    )
    client.upload_points(
        collection_name="fashion_knowledge",
        points=[models.PointStruct(id=i, vector=vector.tolist(), payload=chunk) for i, (vector, chunk) in enumerate(zip(vectors, chunks))],
    )
    return client

In [ ]:
chunks = load_chunks()
qdrant = build_index(chunks)


def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_#-]+|[\u4e00-\u9fff]", text.lower())

corpus = [tokenize(item["text"]) for item in chunks]
doc_frequency = Counter(token for tokens in corpus for token in set(tokens))
avg_length = sum(len(tokens) for tokens in corpus) / len(corpus)

def bm25_search(question, top_k=10, k1=1.5, b=0.75):
    scores = []
    for tokens in corpus:
        counts = Counter(tokens)
        score = 0.0
        for token in tokenize(question):
            if token not in counts:
                continue
            df = doc_frequency[token]
            idf = math.log(1 + (len(corpus) - df + 0.5) / (df + 0.5))
            tf = counts[token]
            norm = tf + k1 * (1 - b + b * len(tokens) / avg_length)
            score += idf * tf * (k1 + 1) / norm
        scores.append(score)
    indexes = np.argsort(scores)[::-1][:top_k]
    return [(index, float(scores[index])) for index in indexes]

In [ ]:
def rrf(dense_results, sparse_results, top_k=4, constant=60):
    scores = defaultdict(float)
    for rank, (index, _) in enumerate(dense_results, start=1):
        scores[index] += 1 / (constant + rank)
    for rank, (index, _) in enumerate(sparse_results, start=1):
        scores[index] += 1 / (constant + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]

question = "SKU-JK902 Cordura 500D 的规格是什么？"
question_vector = model.encode(question, normalize_embeddings=True).tolist()
dense_hits = qdrant.query_points(collection_name="fashion_knowledge", query=question_vector, limit=10).points
dense_results = [(int(hit.id), float(hit.score)) for hit in dense_hits]
sparse_results = bm25_search(question)
final_results = rrf(dense_results, sparse_results)

for rank, (index, score) in enumerate(final_results, start=1):
    print(f"{rank}. {score:.4f}  {chunks[index]['source']}")

混合检索不是把两个分数硬加，而是先把两路结果变成排名，再用 RRF 合并。这样不要求两种算法的原始分数处在同一个尺度。

## 对照 P09：三种排名放在一起

同一个问题、同一批片段，分别看纯向量、BM25 和 RRF 的目标排名。只有这样，才能验证混合检索是否真的补上了货号命中能力。

In [ ]:
target_source = "产品/冲锋衣-JK902/产品规格.md"
target_keyword = "SKU-JK902"
def target_index(results):
    for rank, (index, _) in enumerate(results, start=1):
        if chunks[index]["source"] == target_source and target_keyword in chunks[index]["text"]:
            return rank
    return None

dense_results_all = [
    (int(hit.id), float(hit.score))
    for hit in qdrant.query_points(
        collection_name="fashion_knowledge", query=question_vector, limit=len(chunks)
    ).points
]
bm25_results_all = bm25_search(question, top_k=len(chunks))
rrf_results_all = rrf(dense_results_all, bm25_results_all, top_k=len(chunks))

print(f"查询：{question}")
print(f"目标片段排名 | Dense: {target_index(dense_results_all)} | BM25: {target_index(bm25_results_all)} | RRF: {target_index(rrf_results_all)}")
for name, results in [("Dense", dense_results_all), ("BM25", bm25_results_all), ("RRF", rrf_results_all)]:
    print(f"\n{name} Top-5")
    for rank, (index, score) in enumerate(results[:5], start=1):
        print(f"{rank}. {score:.4f}  {chunks[index]['source']}")

## 用同一组商品验证混合检索

P09 只看稠密检索的局限。这里用同一组商品、同一个查询，对比稠密检索、BM25 和 RRF。

In [ ]:
demo_catalog = [
    {"text": "商品名称：iPhone Pro4；型号：IPHONE-PRO4；定位：专业影像手机。", "name": "iPhone Pro4"},
    {"text": "商品名称：苹果 Pro4；型号：APPLE-PRO4；定位：专业影像手机。", "name": "苹果 Pro4"},
    {"text": "商品名称：苹果 Plug5；型号：APPLE-PLUG5；定位：便携手机配件。", "name": "苹果 Plug5"},
]
demo_query = "iPhone Pro4"
demo_vectors = model.encode([item["text"] for item in demo_catalog], normalize_embeddings=True)
demo_query_vector = model.encode(demo_query, normalize_embeddings=True)
demo_dense = demo_vectors @ demo_query_vector
dense_demo = sorted(enumerate(demo_dense), key=lambda item: item[1], reverse=True)

def simple_bm25(texts, query, k1=1.5, b=0.75):
    tokenized = [tokenize(text) for text in texts]
    query_tokens = tokenize(query)
    frequencies = Counter(token for tokens in tokenized for token in set(tokens))
    average = sum(len(tokens) for tokens in tokenized) / len(tokenized)
    scores = []
    for tokens in tokenized:
        counts = Counter(tokens)
        score = 0.0
        for token in query_tokens:
            if token not in counts:
                continue
            df = frequencies[token]
            idf = math.log(1 + (len(tokenized) - df + 0.5) / (df + 0.5))
            tf = counts[token]
            norm = tf + k1 * (1 - b + b * len(tokens) / average)
            score += idf * tf * (k1 + 1) / norm
        scores.append(score)
    return sorted(enumerate(scores), key=lambda item: item[1], reverse=True)

demo_sparse = simple_bm25([item["text"] for item in demo_catalog], demo_query)
demo_rrf = rrf(demo_dense, demo_sparse, top_k=len(demo_catalog))
for label, results in [("Dense", demo_dense), ("BM25", demo_sparse), ("RRF", demo_rrf)]:
    print(label, "->", [demo_catalog[index]["name"] for index, _ in results])

这个例子说明：稠密检索负责找主题相近的候选，BM25 负责确认精确词项，RRF 再把两路排名合并。混合检索的价值不是让所有结果都正确，而是降低单一路径漏掉正确实体的风险。